[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Losses and Optimizers &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os
import warnings

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' tensors, rebuilt here so that each answer runs on its own.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
X_test = torch.from_numpy(scaled[test_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
snow_train = torch.from_numpy(snow_on_may_1[train_rows]).unsqueeze(1)
snow_test = torch.from_numpy(snow_on_may_1[test_rows]).unsqueeze(1)
band = np.digitize(jan_mean, [-8.0, -3.0])

**1.** Compute the mean absolute error of `predicted` and `measured` with `torch.nn.L1Loss` and with `keras.losses.MeanAbsoluteError`, and check both against the mean of the absolute differences.

In [3]:
predicted = torch.tensor([[-3.0], [-7.5], [0.5]])
measured = torch.tensor([[-2.0], [-8.0], [1.5]])

print("by hand:", round((predicted - measured).abs().mean().item(), 4))
print("PyTorch:", round(torch.nn.L1Loss()(predicted, measured).item(), 4))
print("Keras:  ", round(float(keras.losses.MeanAbsoluteError()(measured.numpy(), predicted.numpy())), 4))

by hand: 0.8333
PyTorch: 0.8333
Keras:   0.8333


The absolute errors are 1, 0.5 and 1 degree, so all three give 0.8333. Keras's loss takes the measured values first, as its mean squared error does.

**2.** Find the logit whose probability is 0.9 with `torch.logit`, and check it with `torch.sigmoid`.

In [4]:
logit = torch.logit(torch.tensor(0.9))
print(round(logit.item(), 4), round(torch.sigmoid(logit).item(), 4))

2.1972 0.9


A logit of 2.1972 is a probability of 0.9. `torch.logit` is the inverse of the sigmoid, the log of p / (1 − p).

**3.** Train the snow classifier with `torch.optim.Adam` at a learning rate of 0.01 for 500 steps, from `torch.manual_seed(0)`, and print its accuracy on the test stations.

In [5]:
torch.manual_seed(0)
snow_model = torch.nn.Linear(4, 1)
optimizer = torch.optim.Adam(snow_model.parameters(), lr=0.01)
bce = torch.nn.BCEWithLogitsLoss()
for step in range(500):
    optimizer.zero_grad()
    loss = bce(snow_model(X_train), snow_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    accuracy = ((snow_model(X_test) > 0).float() == snow_test).float().mean().item()
print("training loss", round(loss.item(), 3), "| test accuracy", round(accuracy, 3))

training loss 0.328 | test accuracy 0.805


Adam reaches a training loss of 0.328 and a test accuracy of 0.805, against 0.329 and 0.8 for plain gradient descent at 0.1, one test station apart. For a model this small, both end in about the same place.

**4.** Use `losses_by_step` to try plain gradient descent at a learning rate of 0.1 with `momentum=0.9`, and compare its losses with the table's.

In [6]:
def losses_by_step(make_optimizer, steps=(10, 50, 100, 200), seed=0):
    """Train a fresh 4-32-1 network on the January means and return its loss at each of steps."""
    torch.manual_seed(seed)
    net = torch.nn.Sequential(torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 1))
    optimizer = make_optimizer(net.parameters())
    losses = []
    for step in range(1, max(steps) + 1):
        optimizer.zero_grad()
        loss = torch.nn.MSELoss()(net(X_train), y_train)
        loss.backward()
        optimizer.step()
        if step in steps:
            losses.append(round(loss.item(), 3))
    return losses


momentum = lambda p: torch.optim.SGD(p, lr=0.1, momentum=0.9)
adam = lambda p: torch.optim.Adam(p, lr=0.1)
print("                            step 10   50  100  200")
print("SGD, 0.1, momentum, seed 0", losses_by_step(momentum, seed=0))
print("SGD, 0.1, momentum, seed 1", losses_by_step(momentum, seed=1))
print("Adam, 0.1, seed 1         ", losses_by_step(adam, seed=1))

                            step 10   50  100  200
SGD, 0.1, momentum, seed 0 [2.64, 0.802, 0.743, 0.711]
SGD, 0.1, momentum, seed 1 [7.496, 0.899, 0.762, 0.72]
Adam, 0.1, seed 1          [3.735, 0.847, 0.748, 0.725]


From seed 0, momentum at 0.1 is lower than every row of the table from 50 steps on. From seed 1 it is behind Adam at 0.1 at 50 and 100 steps, so the ranking belongs to the seed. With momentum 0.9, steps that point the same way build up to about ten times the learning rate.

**5.** Train the three-band classifier in Keras with `keras.losses.CategoricalCrossentropy(from_logits=True)`, which takes a column per band, with labels from `keras.utils.to_categorical(band, 3)`, and print its accuracy on the test stations.

In [7]:
one_hot = keras.utils.to_categorical(band, 3)
print(band[:3], one_hot[:3])

keras.utils.set_random_seed(0)
band_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(3)])
band_keras.compile(optimizer=keras.optimizers.SGD(0.5),
                   loss=keras.losses.CategoricalCrossentropy(from_logits=True),
                   metrics=["categorical_accuracy"])
band_keras.fit(scaled[train_rows], one_hot[train_rows], epochs=500, batch_size=800, verbose=0)

test_loss, test_accuracy = band_keras.evaluate(scaled[test_rows], one_hot[test_rows], verbose=0)
print("test accuracy", round(test_accuracy, 3))

[0 1 1] [[1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]]
test accuracy 0.865


A one-hot label has a 1 in the column of the station's band and 0 elsewhere. The accuracy is the same 0.865 as the sparse form gives, since the two losses compute the same thing from two ways of writing the answer.

---

&#8592; **Back to:** [Losses and Optimizers](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/08-losses-and-optimizers.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
